In [ ]:
from tqdm import tqdm
import requests as r
import pandas as pd
import numpy as np
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import string
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from keras_tuner.tuners import RandomSearch
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score,roc_curve,precision_recall_curve

In [ ]:

API_KEY = "8265bd1679663a7ea12ac168da84d2e8" 
MOVIE_URL = "https://api.themoviedb.org/3/movie/top_rated"
GENRE_URL = "https://api.themoviedb.org/3/genre/movie/list"


In [ ]:
genre_response=r.get(GENRE_URL,params={'api_key':API_KEY,'language':'en-US'})
genre_json=genre_response.json()
genre_map={g['id']:g['name'] for g in genre_json['genres']}
genre_map

In [ ]:
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1,min=1,max=10),
    retry=retry_if_exception_type(r.exceptions.RequestException),
    reraise=True
)
def fetch_page(page):
    response=r.get(MOVIE_URL,params={"api_key":API_KEY,
                                     "language":"en-US",
                                     "page":page},timeout=10)
    
    if response.status_code == 429:
        raise r.exceptions.RequestException("Rate limited")

    response.raise_for_status()
    return response.json()

In [ ]:
all_movies=[]
for page in tqdm(range(1,501)):
   try:
    data=fetch_page(page)

    for m in data['results']:
            all_movies.append({
                "id":m['id'],
                'title':m['title'],
                "description":m['overview'],
                "genre_id":m["genre_ids"]
            })

    time.sleep(0.25)
   except Exception as e:
      print(f"{page} failed",e)
   

In [ ]:
def convert_genres(genre_ids):
    return ", ".join([genre_map.get(g, "Unknown") for g in genre_ids])

for movie in all_movies:
    movie["genres"] = convert_genres(movie["genre_id"])
    del movie["genre_id"]


In [ ]:
df = pd.DataFrame(all_movies)
df.head()

In [ ]:
nltk.download("stopwords",quiet=False)
nltk.download("wordnet")
nltk.download("punkt", download_dir="C:/nltk_data")
nltk.download("punkt_tab", download_dir="C:/nltk_data")

nltk.data.path.append("C:/nltk_data")

nltk.data.path.append("C:/nltk_data")


lemm=WordNetLemmatizer()
stop_words=set(stopwords.words("english"))

def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

exclude=string.punctuation

def remove_punc(text):
    return text.translate(str.maketrans('', '', exclude))


def tokenize(text):
    tokens=word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens=[lemm.lemmatize(w) for w in tokens]
    return " ".join(tokens)

df['description']=df['description'].str.lower()
df['description']=df['description'].apply(remove_html_tags)
df['description']=df['description'].apply(remove_punc)
df['description']=df['description'].apply(tokenize)



In [ ]:
df["genres"] = df["genres"].fillna("")
df = df[df["genres"] != ""]

# convert "Action, Drama" → ["Action", "Drama"]
df["genre_list"] = df["genres"].apply(lambda x: x.split(", "))

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["genre_list"])

genre_columns = mlb.classes_


In [ ]:
genre_columns

In [ ]:
num_genres = y.shape[1]
num_genres

In [ ]:
vectorizer=TfidfVectorizer(max_features=10000)
X=vectorizer.fit_transform(df["description"])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.toarray()
X_test  = X_test.toarray()


In [ ]:
num_features = X_train.shape[1]
num_genres   = y_train.shape[1]

In [ ]:
def build_model(hp):
    model = Sequential()

    
    model.add(Input(shape=(num_features,)))

    for i in range(hp.Int("num_layers", 1, 6)):
        model.add(Dense(
            units=hp.Int(f"units_{i}", min_value=32, max_value=256, step=32),
            activation=hp.Choice(f"activation_{i}", ["relu", "tanh"])
        ))
        model.add(Dropout(hp.Float(f"dropout_{i}", 0.1, 0.5, step=0.1)))

   
    model.add(Dense(num_genres, activation="sigmoid"))

    model.compile(optimizer=hp.Choice("optimizer",values=['adam','rmsprop','adagrad','adadelta','sgd']),
                  loss="binary_crossentropy",
                  metrics=[
                        tf.keras.metrics.BinaryAccuracy(name="bin_acc"),
                        tf.keras.metrics.Precision(name="precision"),
                        tf.keras.metrics.Recall(name="recall"),
                        tf.keras.metrics.AUC(name="auc")
]
)
    return model

In [ ]:
tuner = RandomSearch(
    build_model,
    objective="val_auc",
    max_trials=5,
    directory="kt_dir",
    project_name="text_preprocessing",
    overwrite=True
)


In [ ]:
tuner.search(X_train,y_train,epochs=50,validation_data=(X_test,y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model=tuner.get_best_models(num_models=1)[0]
print(model.summary())

In [ ]:
history=model.fit(X_train,y_train,epochs=20,validation_data=(X_test,y_test),verbose=False)

In [ ]:
y_prob = model.predict(X_test)
auc_micro = roc_auc_score(y_test, y_prob, average="micro")
auc_macro = roc_auc_score(y_test, y_prob, average="macro")

print("Micro AUC:", auc_micro)
print("Macro AUC:", auc_macro)


In [ ]:
plt.plot(history.history['auc'])
plt.plot(history.history['val_auc'])

In [ ]:
clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=5000,
        C=2.0,
        n_jobs=-1
    )
)

clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)

micro_auc = roc_auc_score(y_test, y_prob, average="micro")
macro_auc = roc_auc_score(y_test, y_prob, average="macro")

print("Micro AUC:", micro_auc)
print("Macro AUC:", macro_auc)


In [ ]:
precision, recall, _ = precision_recall_curve(
    y_test.ravel(),
    y_prob.ravel()
)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Micro-average Precision–Recall Curve")
plt.show()


In [ ]:
y_prob = clf.predict_proba(X_test)

fpr, tpr, _ = roc_curve(y_test.ravel(), y_prob.ravel())

plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Micro-average ROC Curve")
plt.show()


In [ ]:
genre = -3
result=[]
coefs = clf.estimators_[genre].coef_[0]
top = np.argsort(coefs)[-20:]

for i in top:
    result.append(vectorizer.get_feature_names_out()[i])

print(result)